# Session 07 · Cleaning of Imported Data

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Example 1 - inspect a messy survey

In [1]:
import numpy as np
import pandas as pd
raw = pd.DataFrame({
    " Name ": ["Ali", "sana ", "Hamza", "Ali", None, "Zoya"],
    "City": ["khi", "Lahore", "KARACHI ", "khi", "Isb", "lahore"],
    "Age": ["21", "abc", "19", "21", "230", "22"],
    "Fee (Rs)": ["Rs. 12,000", "15000", None, "Rs. 12,000", "9,500", None],
})
print(raw.isna().sum())
print("duplicates:", raw.duplicated().sum())

 Name       1
City        0
Age         0
Fee (Rs)    2
dtype: int64
duplicates: 1


## Example 2 - tidy column names and types

In [2]:
df = raw.rename(columns=lambda c: c.strip().lower().replace(" (rs)", "").replace(" ", "_"))
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["fee"] = pd.to_numeric(df["fee"].str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
print(df.dtypes)

name        str
city        str
age     float64
fee     float64
dtype: object


## Example 3 - clean text and categories

In [3]:
df["name"] = df["name"].str.strip().str.title()
city_map = {"khi": "Karachi", "karachi": "Karachi", "isb": "Islamabad", "lahore": "Lahore"}
df["city"] = df["city"].str.strip().str.lower().map(city_map)
print(df)

    name       city    age       fee
0    Ali    Karachi   21.0      0.12
1   Sana     Lahore    NaN  15000.00
2  Hamza    Karachi   19.0       NaN
3    Ali    Karachi   21.0      0.12
4    NaN  Islamabad  230.0   9500.00
5   Zoya     Lahore   22.0       NaN


## Example 4 - impossible values, duplicates, missing

In [4]:
df.loc[~df["age"].between(10, 80), "age"] = np.nan
df = df.drop_duplicates().dropna(subset=["name"]).copy()
df["age"] = df["age"].fillna(df["age"].median())
df["fee"] = df["fee"].fillna(df.groupby("city")["fee"].transform("median"))
print(df)

    name     city   age       fee
0    Ali  Karachi  21.0      0.12
1   Sana   Lahore  21.0  15000.00
2  Hamza  Karachi  19.0      0.12
5   Zoya   Lahore  22.0  15000.00


## Example 5 - outliers with IQR and z-score

In [5]:
rng = np.random.default_rng(7)
s = pd.Series(np.append(rng.normal(50, 5, 200), [95, 4]))
q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
iqr_out = s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]
z = (s - s.mean()) / s.std()
print("IQR outliers:", iqr_out.round(1).tolist())
print("z>3 outliers:", s[z.abs() > 3].round(1).tolist())
capped = s.clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
print("max before/after capping:", round(s.max(), 1), round(capped.max(), 1))

IQR outliers: [37.4, 61.2, 95.0, 4.0]
z>3 outliers: [95.0, 4.0]
max before/after capping: 95.0 61.1


## Example 6 - validate

In [6]:
assert df["name"].notna().all()
assert df["age"].between(10, 80).all()
assert set(df["city"].dropna()) <= {"Karachi", "Lahore", "Islamabad"}
assert not df.duplicated().any()
print("Clean data passed all checks")

Clean data passed all checks
